# Experimentos 1 e 2 V3 — Zero-Shot e Few-Shot (Modelos Pequenos)

**Modelos:** Mistral 7B, Qwen 2.5 7B, Gemma 2 9B  
**Split:** 80/20 | **Folds:** 5 com media e desvio padrao  
**Otimizacao:** cada modelo carregado uma vez — ZS e FS rodados antes de liberar GPU.  
**Checkpoints:** salvos a cada 50 redacoes por fold. Retomada automatica ao reexecutar.

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft optuna
print('Instalado!')

In [ ]:
# Celula 2 - Imports e autenticacao
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN     = userdata.get('HF_TOKEN')
N_FOLDS      = 5
RANDOM_STATE = 42

login(token=HF_TOKEN)
print('Autenticado!')

In [ ]:
# Celula 3 - Dataset e K-Fold (80/20)
import gdown
import pandas as pd
import re
import ast
from sklearn.model_selection import StratifiedKFold

novo_id = '1chJZo8L4s3Zuv1nzHrZa3b6ycf0ePQhv'
gdown.download(
    f'https://drive.google.com/uc?id={novo_id}',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')

df_enem[['c1', 'c2', 'c3', 'c4', 'c5']] = df_enem['competence'].apply(ast.literal_eval).tolist()


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

print(f'Total: {len(df_enem)} | Folds: {N_FOLDS} | Teste por fold: ~{len(df_enem) // N_FOLDS}')

In [ ]:
# Celula 4 - Funcoes auxiliares

def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


def agregar_folds(resultados_folds, nome):
    validos = [r for r in resultados_folds if r is not None]
    if not validos: return None
    metricas = ['mae', 'rmse', 'qwk', 'f1']
    agregado = {'modelo': nome}
    for m in metricas:
        vals = [r[m] for r in validos if not np.isnan(r[m])]
        agregado[m]          = float(np.mean(vals)) if vals else float('nan')
        agregado[m + '_std'] = float(np.std(vals))  if vals else float('nan')
    agregado['n_folds'] = len(validos)
    print(f'\n{"="*60}')
    print(f'MEDIA {N_FOLDS} FOLDS — {nome}')
    print(f'{"="*60}')
    for m in metricas:
        print(f'  {m.upper():<6}: {agregado[m]:.4f} +/- {agregado[m+"_std"]:.4f}')
    print(f'  Folds validos: {agregado["n_folds"]}/{N_FOLDS}')
    print(f'{"="*60}')
    return agregado


print('Funcoes auxiliares carregadas!')

In [ ]:
# Celula 5 - Prompts e inferencia

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)


def selecionar_exemplos_fs(df_treino):
    ex_baixo = df_treino[df_treino['score'].between(80, 300)].iloc[0]
    ex_medio = df_treino[df_treino['score'].between(400, 600)].iloc[0]
    ex_alto  = df_treino[df_treino['score'].between(700, 1000)].iloc[0]
    return [ex_baixo, ex_medio, ex_alto]


def gerar_prompt_zs(redacao, max_chars=None):
    if max_chars: redacao = redacao[:max_chars]
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala oficial: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


def gerar_prompt_fs(redacao, exemplos, max_chars=None, n_exemplos=3):
    if max_chars: redacao = redacao[:max_chars]
    bloco = ''
    for j, ex in enumerate(exemplos[:n_exemplos]):
        notas_ex = (
            '{"C1": ' + str(ex['c1']) + ', "C2": ' + str(ex['c2']) +
            ', "C3": ' + str(ex['c3']) + ', "C4": ' + str(ex['c4']) +
            ', "C5": ' + str(ex['c5']) + ', "Nota_Total": ' + str(ex['score']) + '}'
        )
        bloco += (
            '--- EXEMPLO ' + str(j + 1) + ' (score=' + str(ex['score']) + ') ---\n'
            'REDACAO: "' + ex['essay_limpo'][:300] + '..."\n'
            'AVALIACAO: ' + notas_ex + '\n\n'
        )
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie usando a escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): 0-200\n- C2 (Tema/Estrutura): 0-200\n'
        '- C3 (Argumentacao): 0-200\n- C4 (Coesao): 0-200\n'
        '- C5 (Proposta de Intervencao): 0-200\n\n'
        'EXEMPLOS AVALIADOS POR HUMANOS:\n' + bloco +
        'REDACAO A AVALIAR:\n"' + redacao + '"\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


def carregar_modelo(nome_modelo):
    print(f'Carregando {nome_modelo}...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    m.eval()
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


def inf_local(tok, m, prompt, temp=0.1, max_tok=300):
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print('Prompts e inferencia carregados!')

In [ ]:
# Celula 6 - Optuna e runners ZS/FS por fold

def ckpt_path(nome_curto, exp, fold):
    return f'ck_exp{exp}_{nome_curto}_fold{fold}.csv'


def fold_completo(nome_curto, exp, fold):
    path = ckpt_path(nome_curto, exp, fold)
    if not os.path.exists(path): return False
    df_ck = pd.read_csv(path)
    n_te  = len(df_enem[df_enem['fold'] == fold])
    return len(df_ck.dropna(subset=['pred_total'])) >= n_te * 0.95


def obj_optuna_zs(trial, fn_inf):
    temp      = trial.suggest_float('temp', 0.01, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)
    df_val    = df_enem[df_enem['fold'] == 1].head(10)
    yp, yt    = [], []
    for _, row in df_val.iterrows():
        try:
            resp  = fn_inf(gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars), temp)
            notas = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def obj_optuna_fs(trial, fn_inf):
    temp       = trial.suggest_float('temp', 0.01, 0.5)
    max_chars  = trial.suggest_int('max_chars', 500, 2000, step=250)
    n_exemplos = trial.suggest_int('n_exemplos', 1, 3)
    df_val     = df_enem[df_enem['fold'] == 1].head(10)
    exemplos_v = selecionar_exemplos_fs(df_enem[df_enem['fold'] != 1])
    yp, yt     = [], []
    for _, row in df_val.iterrows():
        try:
            prompt = gerar_prompt_fs(row['essay_limpo'], exemplos_v, max_chars=max_chars, n_exemplos=n_exemplos)
            resp   = fn_inf(prompt, temp)
            notas  = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_fold(nome, nome_curto, fn_inf, params, exp, fold, df_te, tipo='ZS', exemplos=None):
    temp      = params.get('temp', 0.1)
    max_chars = params.get('max_chars', None)
    n_ex      = params.get('n_exemplos', 3)
    path      = ckpt_path(nome_curto, exp, fold)

    if os.path.exists(path):
        df_ck     = pd.read_csv(path)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'  Checkpoint fold {fold}: {len(ja_feitos)} ja processadas')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total'])
        ja_feitos = set()

    pendentes = df_te[~df_te.index.isin(ja_feitos)]
    print(f'  Pendentes fold {fold}: {len(pendentes)}/{len(df_te)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            if tipo == 'ZS':
                prompt = gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars)
            else:
                prompt = gerar_prompt_fs(row['essay_limpo'], exemplos, max_chars=max_chars, n_exemplos=n_ex)
            resp  = fn_inf(prompt, temp)
            notas = extrair(resp)
            pred  = notas['Nota_Total'] if notas else None
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(path, index=False)
            novos = []
            print(f'    Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_te)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(path, index=False)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'  Validas fold {fold}: {len(df_v)}/{len(df_te)}')
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), f'{nome} fold{fold}')
    return None


print('Runners carregados!')

In [ ]:
# Celula 7 - Mistral 7B (ZS + FS no mesmo carregamento)
nome_modelo  = 'mistralai/Mistral-7B-Instruct-v0.3'
nome_curto   = 'mistral'
nome_display = 'Mistral 7B'

print(f'\n{"="*60}')
print(f'MISTRAL 7B — EXP1 (Zero-Shot) + EXP2 (Few-Shot)')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_inf = lambda p, t: inf_local(tok, m, p, temp=t)

study_zs = optuna.create_study(
    study_name='optuna_mistral_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_mistral_zs_fs.db',
    load_if_exists=True
)
if len(study_zs.trials) < 10:
    study_zs.optimize(lambda t: obj_optuna_zs(t, fn_inf), n_trials=10, catch=(Exception,))
params_zs_mistral = study_zs.best_params
print(f'ZS melhores params: {params_zs_mistral}')

study_fs = optuna.create_study(
    study_name='optuna_mistral_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_mistral_zs_fs.db',
    load_if_exists=True
)
if len(study_fs.trials) < 10:
    study_fs.optimize(lambda t: obj_optuna_fs(t, fn_inf), n_trials=10, catch=(Exception,))
params_fs_mistral = study_fs.best_params
print(f'FS melhores params: {params_fs_mistral}')

res_mistral_zs = []
res_mistral_fs = []
for fold in range(N_FOLDS):
    df_te    = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    df_tr    = df_enem[df_enem['fold'] != fold]
    exemplos = selecionar_exemplos_fs(df_tr)

    if fold_completo(nome_curto, 1, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 1, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} ZS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} ZS', nome_curto, fn_inf, params_zs_mistral, 1, fold, df_te, tipo='ZS')
    res_mistral_zs.append(res)

    if fold_completo(nome_curto, 2, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 2, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} FS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} FS', nome_curto, fn_inf, params_fs_mistral, 2, fold, df_te, tipo='FS', exemplos=exemplos)
    res_mistral_fs.append(res)

liberar(m, tok)

media_mistral_zs = agregar_folds(res_mistral_zs, f'{nome_display} (Zero-Shot)')
media_mistral_fs = agregar_folds(res_mistral_fs, f'{nome_display} (Few-Shot)')
pd.DataFrame([r for r in res_mistral_zs if r]).to_csv(f'resultados_exp1_{nome_curto}_folds.csv', index=False)
pd.DataFrame([r for r in res_mistral_fs if r]).to_csv(f'resultados_exp2_{nome_curto}_folds.csv', index=False)
print(f'\nMistral 7B Exp1+2 completo!')

In [ ]:
# Celula 8 - Qwen 2.5 7B (ZS + FS no mesmo carregamento)
nome_modelo  = 'Qwen/Qwen2.5-7B-Instruct'
nome_curto   = 'qwen7b'
nome_display = 'Qwen 2.5 7B'

print(f'\n{"="*60}')
print(f'QWEN 2.5 7B — EXP1 (Zero-Shot) + EXP2 (Few-Shot)')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_inf = lambda p, t: inf_local(tok, m, p, temp=t)

study_zs = optuna.create_study(
    study_name='optuna_qwen7b_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_qwen7b_zs_fs.db',
    load_if_exists=True
)
if len(study_zs.trials) < 10:
    study_zs.optimize(lambda t: obj_optuna_zs(t, fn_inf), n_trials=10, catch=(Exception,))
params_zs_qwen7b = study_zs.best_params
print(f'ZS melhores params: {params_zs_qwen7b}')

study_fs = optuna.create_study(
    study_name='optuna_qwen7b_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_qwen7b_zs_fs.db',
    load_if_exists=True
)
if len(study_fs.trials) < 10:
    study_fs.optimize(lambda t: obj_optuna_fs(t, fn_inf), n_trials=10, catch=(Exception,))
params_fs_qwen7b = study_fs.best_params
print(f'FS melhores params: {params_fs_qwen7b}')

res_qwen7b_zs = []
res_qwen7b_fs = []
for fold in range(N_FOLDS):
    df_te    = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    df_tr    = df_enem[df_enem['fold'] != fold]
    exemplos = selecionar_exemplos_fs(df_tr)

    if fold_completo(nome_curto, 1, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 1, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} ZS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} ZS', nome_curto, fn_inf, params_zs_qwen7b, 1, fold, df_te, tipo='ZS')
    res_qwen7b_zs.append(res)

    if fold_completo(nome_curto, 2, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 2, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} FS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} FS', nome_curto, fn_inf, params_fs_qwen7b, 2, fold, df_te, tipo='FS', exemplos=exemplos)
    res_qwen7b_fs.append(res)

liberar(m, tok)

media_qwen7b_zs = agregar_folds(res_qwen7b_zs, f'{nome_display} (Zero-Shot)')
media_qwen7b_fs = agregar_folds(res_qwen7b_fs, f'{nome_display} (Few-Shot)')
pd.DataFrame([r for r in res_qwen7b_zs if r]).to_csv(f'resultados_exp1_{nome_curto}_folds.csv', index=False)
pd.DataFrame([r for r in res_qwen7b_fs if r]).to_csv(f'resultados_exp2_{nome_curto}_folds.csv', index=False)
print(f'\nQwen 2.5 7B Exp1+2 completo!')

In [ ]:
# Celula 9 - Gemma 2 9B (ZS + FS no mesmo carregamento)
nome_modelo  = 'google/gemma-2-9b-it'
nome_curto   = 'gemma'
nome_display = 'Gemma 2 9B'

print(f'\n{"="*60}')
print(f'GEMMA 2 9B — EXP1 (Zero-Shot) + EXP2 (Few-Shot)')
print(f'{"="*60}')

tok, m = carregar_modelo(nome_modelo)
fn_inf = lambda p, t: inf_local(tok, m, p, temp=t)

study_zs = optuna.create_study(
    study_name='optuna_gemma_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_gemma_zs_fs.db',
    load_if_exists=True
)
if len(study_zs.trials) < 10:
    study_zs.optimize(lambda t: obj_optuna_zs(t, fn_inf), n_trials=10, catch=(Exception,))
params_zs_gemma = study_zs.best_params
print(f'ZS melhores params: {params_zs_gemma}')

study_fs = optuna.create_study(
    study_name='optuna_gemma_zs_fs',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    storage='sqlite:///optuna_gemma_zs_fs.db',
    load_if_exists=True
)
if len(study_fs.trials) < 10:
    study_fs.optimize(lambda t: obj_optuna_fs(t, fn_inf), n_trials=10, catch=(Exception,))
params_fs_gemma = study_fs.best_params
print(f'FS melhores params: {params_fs_gemma}')

res_gemma_zs = []
res_gemma_fs = []
for fold in range(N_FOLDS):
    df_te    = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    df_tr    = df_enem[df_enem['fold'] != fold]
    exemplos = selecionar_exemplos_fs(df_tr)

    if fold_completo(nome_curto, 1, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 1, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} ZS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} ZS', nome_curto, fn_inf, params_zs_gemma, 1, fold, df_te, tipo='ZS')
    res_gemma_zs.append(res)

    if fold_completo(nome_curto, 2, fold):
        df_ck = pd.read_csv(ckpt_path(nome_curto, 2, fold)).dropna(subset=['pred_total'])
        res   = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'{nome_display} FS fold{fold}')
    else:
        res = rodar_fold(f'{nome_display} FS', nome_curto, fn_inf, params_fs_gemma, 2, fold, df_te, tipo='FS', exemplos=exemplos)
    res_gemma_fs.append(res)

liberar(m, tok)

media_gemma_zs = agregar_folds(res_gemma_zs, f'{nome_display} (Zero-Shot)')
media_gemma_fs = agregar_folds(res_gemma_fs, f'{nome_display} (Few-Shot)')
pd.DataFrame([r for r in res_gemma_zs if r]).to_csv(f'resultados_exp1_{nome_curto}_folds.csv', index=False)
pd.DataFrame([r for r in res_gemma_fs if r]).to_csv(f'resultados_exp2_{nome_curto}_folds.csv', index=False)
print(f'\nGemma 2 9B Exp1+2 completo!')

In [ ]:
# Celula 10 - Consolidacao final
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

todos = [r for r in [
    media_mistral_zs, media_mistral_fs,
    media_qwen7b_zs,  media_qwen7b_fs,
    media_gemma_zs,   media_gemma_fs
] if r is not None]
df_res = pd.DataFrame(todos)

print('\n' + '='*90)
print(f'{"Modelo":<38} {"Folds":>6} {"MAE":>9} {"RMSE":>9} {"QWK":>9} {"F1":>9}')
print('-'*90)
for _, row in df_res.iterrows():
    print(
        f'{row["modelo"]:<38} '
        f'{int(row["n_folds"]):>6} '
        f'{row["mae"]:>7.3f}+/-{row["mae_std"]:.3f} '
        f'{row["rmse"]:>7.3f}+/-{row["rmse_std"]:.3f} '
        f'{row["qwk"]:>7.3f}+/-{row["qwk_std"]:.3f} '
        f'{row["f1"]:>7.3f}+/-{row["f1_std"]:.3f}'
    )
print('='*90)

df_res.to_csv('resultados_exp1_exp2_v3_final.csv', index=False)
print('\nCSV salvo: resultados_exp1_exp2_v3_final.csv')

cores = ['#4C72B0' if 'Zero-Shot' in m else '#DD8452' for m in df_res['modelo']]
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    f'Experimentos 1 e 2 V3 — Zero-Shot vs Few-Shot\nMedia de {N_FOLDS} Folds (80/20)',
    fontsize=14, fontweight='bold'
)
for ax, (titulo, coluna) in zip(axes.flatten(), [
    ('MAE (menor = melhor)', 'mae'),
    ('RMSE (menor = melhor)', 'rmse'),
    ('QWK (maior = melhor)', 'qwk'),
    ('F1 Score (maior = melhor)', 'f1')
]):
    barras = ax.barh(df_res['modelo'], df_res[coluna], color=cores, edgecolor='white')
    ax.errorbar(
        df_res[coluna], range(len(df_res)),
        xerr=df_res[coluna + '_std'],
        fmt='none', color='black', capsize=4, linewidth=1.5
    )
    for b in barras:
        w = b.get_width()
        ax.text(w + 0.002, b.get_y() + b.get_height() / 2,
                f'{w:.3f}', va='center', ha='left', fontsize=8)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

legenda = [Patch(color='#4C72B0', label='Zero-Shot'), Patch(color='#DD8452', label='Few-Shot')]
fig.legend(handles=legenda, loc='lower center', ncol=2, fontsize=11,
           frameon=False, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig('grafico_exp1_exp2_v3_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo: grafico_exp1_exp2_v3_final.png')